# 6 — Phrases, concepts and multi-word expressions

Single words leave the Zipfian regime; phrases do not. This notebook measures
that, then asks whether it survives two obvious objections: that the "phrases"
are surface n-grams rather than linguistic units, and that a concept's position
on the curve is an artefact of how a particular language happens to spell it.

Produces **Figure S3**, **Table S4** and **Table S9**, and the concept ranks that
panel C of Figure 1 is typeset from.

In [ ]:
import os
import subprocess
import sys

REPO = os.path.abspath("..") if os.path.isdir(os.path.join("..", "src")) else os.path.abspath(".")
sys.path.insert(0, os.path.join(REPO, "src"))

import plotting as P


def run(*command, must_succeed=True):
    """Run one pipeline step and, unlike a `!` cell, STOP if it fails.

    An IPython `!` cell throws away the exit status: a step that dies leaves no
    output, no error and no trace, and `nbconvert --execute` still reports the
    notebook as successful. Two defects in this pipeline's history hid exactly
    there, so every step below goes through this instead.

    `must_succeed=False` is used only for the two `--check` diagnostics of
    notebook 1, which print a loud banner rather than stopping the run.
    """
    print(">>", " ".join(str(c) for c in command), flush=True)
    code = subprocess.run([str(c) for c in command], check=False).returncode
    if code and must_succeed:
        raise RuntimeError(f"step failed with exit code {code} - read the output "
                           f"above; nothing after this point is valid")
    if code:
        rule = "*" * 72
        print(rule)
        print(f"*** THIS CHECK FAILED (exit code {code}). Read the output above")
        print("*** before going on: whatever depends on this corpus is missing")
        print("*** or wrong, and so is anything computed from it.")
        print(rule, flush=True)
    return code


def py(script, *args, must_succeed=True):
    """`run` for one of this repository's own scripts."""
    return run(sys.executable, os.path.join(REPO, "src", script), *args,
               must_succeed=must_succeed)


%matplotlib inline
USETEX = P.setup_style()
print("repo:", REPO, "| LaTeX text rendering:", USETEX)

## 6.1 The phrase construction — Table S4

Phrases are built by the random non-overlapping partition of Williams et al.
(2015): the text is cut into parts, each inter-word slot breaking with
probability q, so a k-word phrase seen N times has expected frequency
N q²(1−q)^(k−1). Orders 1–5 are merged with those geometric weights and
normalised **once** — not concatenated raw, which would over-count long phrases.

Two checks run before the table:

* the total phrase mass must equal Σ_k w_k × (number of k-gram positions), the
  exact expectation of the construction;
* the phrase and single-word bands must be **disjoint in every language**, so the
  separation is not an artefact of where the fit window was placed.

The exponent is also stable in q, which is the one free parameter of the
construction: it moves from 0.96–0.99 at q = 0.25 to 1.09–1.14 at q = 0.75.

In [ ]:
import phrase_exponents

ngrams, spec, mass = phrase_exponents.spectra()
mass

In [ ]:
tableS4 = phrase_exponents.exponents(spec)
q_table = phrase_exponents.q_sensitivity(ngrams)
phrase_exponents.write_table(tableS4, q_table)
q_table

In [ ]:
tableS4[["language", "alpha_phrase", "alpha_phrase_lo", "alpha_phrase_hi",
         "zipf_decades_phrase", "alpha_word", "alpha_word_lo", "alpha_word_hi",
         "zipf_decades_word"]]

## 6.2 The annotated concepts — how the list was chosen

`manifests/annotation_phrases.json` holds 26 concepts, each written the
conventional way in each of the five languages, so that a language which
lexicalises a concept as one compound (typically German) can be compared with
one that spells it out as a phrase.

**The list was chosen by hand and it is arbitrary.** It is not a sample from any
defined population, and no claim here depends on it being representative. What
makes it usable is that it was **fixed before any count was looked at**, and that
every concept in it is reported whatever the outcome.

**Exactly one rule removes anything, and it is a rule about period, not about
outcome.** SPGC is dominated by pre-1900 literature, so a concept naming a
20th-century object is being put to the corpus as a question it cannot answer.
The four such concepts — `telephone`, `motor car`, `bus stop` and the reference
word `bus` that goes with `bus stop` — are excluded, leaving the 22 the paper
reports. They are `MODERN` in `src/concept_ranks.py`, applied in one place so
that every table and the figure agree. **Nothing is dropped for lack of
attestation:** all 22 are attested in all five languages, on both the weighted
and the unweighted curve. The four excluded ones stay in the manifest on purpose
— a stated rule applied to a whole list is only checkable if the discarded
entries are visible too. See `docs/DATA.md` §4.

Two limitations are worth stating plainly, because they bound what the numbers
can mean:
* **the five corpora are not translations of one another.** *fauteuil* occurs
  8,437 times against *armchair* 449 partly because French Gutenberg novels talk
  about drawing rooms more than English ones do. No choice of surface forms fixes
  this. It is exactly why the claim reported is about the **region of the curve**
  a concept lands in, and not about equal ranks.

In [ ]:
import json

manifest = json.load(open(os.path.join(REPO, "manifests", "annotation_phrases.json")))
print("concepts in the list:", len(manifest["concepts"]))
for era, items in manifest["_era"].items():
    print(f"  {era:9s} {len(items):2d}  {', '.join(items)}")

## 6.3 Where each concept sits — the numbers behind Figure 1C

The rank of each concept is located on the committed phrase spectrum, i.e. on
exactly the curve Figure 1B plots. Both the raw occurrence count and the rank are
reported, because the Williams weight penalises a k-gram by 2^(k−1) relative to a
one-word compound and a reader has to be able to see that factor separately.

In [ ]:
import phrase_ranks

ann = phrase_ranks.tables()
print(ann["coverage"].to_string())
ann["concept_ranks"][[f"rank_{c}" for c in P.LEGEND_ORDER]]

## 6.4 Figure S3 — dispersion, and what it is worth

The statistic reported is the **dispersion** of the same concept across the five
languages, against a null. Two nulls are given, because the choice matters and a
single one would be a choice made silently:

* five *different* concepts drawn from the same 22 — the conservative null, since
  the 22 occupy only 5.1 of the panel's 8.3 decades and cluster near its middle;
* uniform over the whole panel — the loose null.

Correlations across languages are deliberately **not** reported. The English
surface forms in the manifest include period variants (*looking glass*,
*railway station*), which suppresses every correlation involving English while
leaving the dispersion statistic essentially unchanged. Dispersion is the robust
quantity here and it is the one the paper states.

In [ ]:
import concept_ranks

# build() returns one row per (language, concept). reported_ranks() applies the
# paper's selection -- attested in all five languages, and not anachronistic for
# a nineteenth-century corpus -- which leaves 22 of the 26 in the manifest. The
# set is defined there and nowhere else, so every table and the figure agree.
df, span = concept_ranks.build()
rank, _, kk, order, dropped = concept_ranks.reported_ranks(df)
print(f"{len(order)} of {df.concept.nunique()} concepts reported; "
      f"dropped {len(dropped)}: {', '.join(dropped)}")
concept_ranks.dispersion(rank).round(2)

The full analysis — both nulls, the pairwise matrices and the figure — is run by
the two modules below. `concept_ranks.py` writes the dispersion and null tables;
`figure_SI3.py` draws Figure S3.

In [ ]:
py("concept_ranks.py")

In [ ]:
py("figure_SI3.py")

## 6.5 Would annotated multi-word expressions change the answer? — Table S9

A frequent n-gram is not a lexicalised multi-word expression, so the phrase curve
could in principle be an artefact of counting surface strings. The only manually
annotated resource covering all five languages is PARSEME 1.1, which annotates
**verbal** MWEs in gold corpora of ~10^5 tokens — against the 10^8 of SPGC.

Measured naively the annotated MWEs give α ≈ 0.5–0.8 rather than ≈ 1.0, which
looks like a discrepancy. It is a finite-size effect, and the table separates the
two causes by measuring four curves that share one construction and differ only
in what is counted or how much text is available:

1. annotated MWEs, PARSEME gold;
2. surface 2–5-grams from the **same** PARSEME text — same corpus, same size;
3. surface 2–5-grams from SPGC truncated to the same token count;
4. surface 2–5-grams from the whole SPGC.

Rows 1 and 2 agreeing is the answer: on the same text it makes no measurable
difference whether the units are annotated or read off the surface. The distance
to row 4 is corpus size, plus the fact that Figure 1B partitions the *whole* text
so one-word phrases carry the largest Williams weight.

Needs `parseme_mwe.npz` from notebook 2.

In [ ]:
py("mwe_ranks.py")

**Next:** `07_learners_vs_natives.ipynb`.